# Funcionalidades de Pyspark

`````{admonition} Objetivo de la sesión. 
:class: tip
En esta sesión aprenderás los fundamentos de PySpark, una poderosa herramienta para el procesamiento distribuido de datos. Exploraremos cómo crear sesiones de Spark, manipular DataFrames y realizar operaciones comunes con ejemplos prácticos
`````

## ¿Qué es PySpark?

 * PySpark es la interfaz de Python para Apache Spark, un motor de procesamiento de datos en clústeres. Permite trabajar con grandes volúmenes de datos de forma paralela y distribuida.

### ¿Por qué usar PySpark?

 * Procesa datos a gran escala (Big Data).
 * Compatible con múltiples fuentes: CSV, JSON, Parquet, bases de datos.
 * Ideal para tareas de ETL, análisis de datos y machine learning.

### ¿Qué es un `SparkSession`?

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MiPrimeraSesion") \
    .getOrCreate()
```

 * SparkSession es el punto de entrada principal para trabajar con PySpark.
 * Permite crear DataFrames, leer archivos, ejecutar consultas SQL, etc.
 * Solo necesitas una instancia por aplicación.


### ¿Qué es un DataFrame en PySpark?

Un DataFrame es una estructura de datos tabular, similar a una tabla de base de datos o un DataFrame de pandas, pero distribuido.

```python
df = spark.read.csv("datos/xxxxxx.csv", header=True, inferSchema=True)
df.show(5)
```

 * Columnas con nombres y tipos de datos
 * Operaciones distribuidas y optimizadas
 * Inmutable (cada transformación genera un nuevo DataFrame)

### ¿Qué es un DataFrame en PySpark?

```python
# CSV
df_csv = spark.read.csv("datos/xxxxxx.csv", header=True, inferSchema=True)
# Parquet
df_parquet = spark.read.parquet("datos/xxxxxx.parquet")
```

 * header=True: usa la primera fila como nombres de columnas
 * inferSchema=True: detecta automáticamente los tipos de datos

### Operaciones básicas

Importando las funciones necesarias:
```python
from pyspark.sql.functions import col, lit, when, avg, count, sum, asc, desc, struct, concat, substring
from pyspark.sql.types import StringType, IntegerType

#Inspección y Preparación del DataFrame

df.printSchema()	             # Muestra el esquema de la tabla (nombres de columnas y sus tipos de datos).
df.show(5, truncate=False)	     # Muestra las primeras 5 filas de la tabla, sin truncar las cadenas largas.
df.count()	                     # Devuelve el número total de filas en el DataFrame.
df.select("ciudad").distinct().show()	# Muestra las etiquetas únicas de todas las categorías contenidas en la columna ciudad.

# Selección, Filtro y Eliminación de Columnas
# Estas son las operaciones de manipulación de datos más comunes.
df.select("nombre", "edad", "ingresos").show()	           # Selecciona las columnas nombre, edad e ingresos y las muestra.
df = df.drop("columna_innecesaria")	                       # Elimina de la tabla la columna con nombre columna_innecesaria.
df = df.filter(df.edad > 30).show()	                       # Filtra la tabla para mostrar solo los registros donde edad sea mayor a 30 años.
df = df.where((df.ciudad == "Madrid") & (df.edad > 25))	   # Filtra (igual que filter()) los registros que cumplen ambas condiciones (ciudad es "Madrid" Y edad es mayor a 25).
df = df.filter(col("pais").isin(["ES", "FR"]))	           # Filtra los registros donde la columna pais es "ES" o "FR".
df.dropna(subset=['email']).show()	                       # Elimina las filas donde el valor de la columna email es nulo (NULL o NaN).
df.fillna({'ingresos': 0, 'ciudad': 'Desconocida'}).show() # Reemplaza los valores nulos (NULL o NaN) con el valor especificado para cada columna.


# Creación y Transformación de Columnas (withColumn)
# Las funciones más poderosas para la ingeniería de características (feature engineering).
df = df.withColumn("edad_doble", col("edad") * 2)	            # Crea la columna edad_doble como resultado de multiplicar la columna edad por 2.
df = df.withColumn("EDAD_STR", col("edad").cast(StringType()))	# Convierte el tipo de datos de la columna edad a formato texto (StringType) y la nombra EDAD_STR.
df = df.withColumnRenamed("fnac", "Fecha_nac")	                # Renombra la columna original fnac para que ahora se llame Fecha_nac.
df = df.withColumn("Categoria_Edad", when(col("edad") < 25, "Joven").when(col("edad") < 35, "Adulto").otherwise("Mayor")) # Crea una nueva columna Categoria_Edad y clasifica los registros                                                                                                                             con lógica condicional (CASE WHEN de SQL).
df = df.withColumn("Ciudad_Pais", concat(col("ciudad"), lit(", "), col("pais"))) # Concatena los valores de ciudad y pais, separándolos por una coma, y crea la nueva columna Ciudad_Pais.
df = df.withColumn("Inicial_Nombre", substring(col("nombre"), 1, 1))	# Extrae el primer carácter de la columna nombre (posición 1, longitud 1) y lo guarda en Inicial_Nombre.
df = df.withColumn("Nombre_Mayus", upper(col("nombre")))	            # Convierte a mayúsculas la cadena en la columna nombre (requiere from pyspark.sql.functions import upper).


# Agrupación, Agregación y Ordenación
# Esenciales para el resumen y el análisis de datos.
df.groupBy("ciudad").agg(avg("ingresos").alias("ingreso_promedio"), count("*").alias("num_personas")).show() # Agrupa por ciudad y calcula el promedio de ingresos y el conteo de personas                                                                                                                 para cada grupo.
df.sort(col("ciudad").asc(), col("edad").desc()).show()	                       # Ordena la tabla: Primero, ascendentemente por ciudad, y dentro de cada ciudad, descendentemente por edad.
df.orderBy("ingresos", ascending=False).show()	                               # Ordena la tabla por la columna ingresos de forma descendente.


# Combinación de DataFrames (Joins)
# Muestra cómo unir dos DataFrames, df_personas y df_ciudades.
df_join = df_personas.join(df_ciudades, on='id_ciudad', how='inner') # Combina df_personas con df_ciudades usando la columna id_ciudad como clave, manteniendo solo las filas que tienen coincidencia en ambos (inner join).
```s formatos.

### Ejemplos de `expr()` y `sql()` en PySpark

La función `expr()` te permite escribir una expresión SQL como una cadena de texto y aplicarla dentro de funciones de DataFrame como `select()` o `withColumn()`. Es la forma más flexible de usar lógica SQL a nivel de columna sin tener que crear una vista temporal.

Ambas funciones requieren importaciones:

```python
from pyspark.sql.functions import expr, col

df.select(expr("nombre, edad * 2 AS edad_doble")).show()	        #Selecciona columnas y realiza una operación de cálculo simple, nombrando el resultado como edad_doble.
df = df.withColumn("ingreso_anual", expr("ingresos_mensual * 12"))	#Crea una nueva columna (ingreso_anual) aplicando una expresión matemática simple directamente en el DataFrame.
df = df.withColumn("Categoria_Nivel", expr("CASE WHEN edad < 20 THEN 'Joven' ELSE 'Adulto' END"))	#Implementa lógica condicional (CASE WHEN) de SQL dentro de un withColumn para crear una columna categórica.
df.filter(expr("ingresos_mensual > 5000 AND ciudad = 'Bogota'")).show()	        # Filtra el DataFrame utilizando una expresión booleana compleja.
df.select(expr("substring(email, 1, instr(email, '@') - 1) AS usuario")).show()	# Utiliza funciones de cadena de SQL (substring, instr) para extraer el nombre de usuario de una dirección de correo electrónico.
df.groupBy(expr("year(fecha_registro)")).agg(count("*")).show()	# Agrupa los registros extrayendo el año de una columna de fecha, todo dentro de la función expr().

```

### Uso de spark.sql() (Consultas SQL Completas

La función spark.sql() te permite ejecutar consultas SQL completas (SELECT, GROUP BY, JOIN, etc.) sobre tablas temporales registradas en el catálogo de Spark. Es ideal para ejecutar lógica de negocio que está bien definida en SQL.


*Crear una Vista Temporal*:

Para usar spark.sql(), primero debes registrar tu DataFrame como una tabla (o vista) temporal.

```python

# Registra el DataFrame 'df' como una vista SQL temporal llamada 'datos_personas'
df.createOrReplaceTempView("datos_personas")
```

### Ejecutar Consultas SQL

```python

df_sql_select = spark.sql("SELECT nombre, edad, ingresos_mensual FROM datos_personas WHERE edad > 25")	# Selecciona y filtra datos directamente con una consulta SQL.
df_sql_group = spark.sql("SELECT ciudad, AVG(ingresos_mensual) AS promedio_ingreso FROM datos_personas GROUP BY ciudad ORDER BY promedio_ingreso DESC")	# Agrupa por ciudad y calcula el promedio de ingresos, ordenando el resultado.
df_sql_join = spark.sql("SELECT p.*, c.pais FROM datos_personas p INNER JOIN df_ciudades c ON p.id_ciudad = c.id_ciudad")	# Ejecuta una operación JOIN completa con sintaxis SQL, asumiendo que df_ciudades también fue registrada como una vista temporal.
df_sql_case = spark.sql("SELECT nombre, CASE WHEN ingresos_mensual > 7000 THEN 'Alto' ELSE 'Bajo' END AS Nivel_Salarial FROM datos_personas")	# Implementa la lógica CASE WHEN en la cláusula SELECT para categorizar los ingresos.
```

### Operaciones básicas

```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType # Invoca funciones para la identificación de estructura del dataframe en pyspark

schema = StructType([
    StructField("nombre", StringType(), True),
    StructField("edad", IntegerType(), True)
])

df = spark.read.csv("datos/personas.csv", header=True, schema=schema) # Declara para cada una de las columnas dentro de la tabla el tipo de formato, para nombre tipo string y edad integer
```

`````{admonition} Precaución. 
:class: warning
Collect
`````